# geocore — geometric computation core

A walkthrough of all four layers, architected with PyTorch as the structural reference:

- **L0** geometric objects (Pauli, Rotation)
- **L1** geometric operator dispatch
- **L2** automatic machine-precision verification (the autograd analogue)
- **L3** reduce computation: closed-form shortcuts with *measured* speedups (the torch.compile analogue)

Engine-presentation separation: the derivation engine is the geometry theory; everything shown here is standard mathematics verified to machine precision.

In [1]:
import numpy as np
import geocore as gc
from geocore import Pauli, Rotation, get_op
from geocore import shortcuts

## L0 — geometric objects

A Pauli is an element of the Clifford algebra with a canonical 2n-bit symplectic encoding. Commutation is decided by the symplectic form; conjugation by Cliffords is a symplectic transformation with the r-bit phase tracked exactly.

In [2]:
# Pauli objects: commutation via the symplectic form
X, Z, Y = Pauli("X"), Pauli("Z"), Pauli("Y")
print("X commutes with X:", X.commutes_with(Pauli("X")))
print("X commutes with Z:", X.commutes_with(Z))

# Conjugation by a Clifford: S.X.S^+ = Y, S.Y.S^+ = -X (phase r=1!)
conj, r = X.conjugate_by([("s", (0,))])
print(f"S.X.S^+ = {conj.axis} (r={r})")
conj2, r2 = Y.conjugate_by([("s", (0,))])
print(f"S.Y.S^+ = {conj2.axis} (r={r2}, minus sign tracked)")

X commutes with X: True
X commutes with Z: False
S.X.S^+ = Y (r=0)
S.Y.S^+ = X (r=1, minus sign tracked)


## L1 — geometric operator dispatch

Operators are first-class geometric mappings between geometric objects, dispatching by geometric type. Each operator documents the geometric theorem it implements.

In [3]:
# dispatch by geometric type
print("commutes:", get_op("pauli.commutes")(Pauli("X"), Pauli("Y")))
m = get_op("rotation.merge")(Rotation("XX", 0.3), Rotation("XX", 0.4))
print("merge (closure of phase addition):", m)
print("cancels (2-pi closure):", Rotation("X", 2*np.pi).cancels(), Rotation("X", 0.5).cancels())

# circuit optimization (fixed-point completeness)
rots = [("XXII", -np.pi/4), ("ZZIY", np.pi/4), ("ZZYI", np.pi/4),
        ("YYXX", np.pi/4), ("XXII", np.pi/4)]
opt, cl = get_op("circuit.optimize")(rots)
print(f"circuit.optimize: {len(rots)} -> {len(opt)} rotations")

commutes: False
merge (closure of phase addition): Rotation('XX', 0.7)
cancels (2-pi closure): True False
circuit.optimize: 5 -> 3 rotations


## L2 — automatic verification (≈ autograd)

PyTorch's core automatically *differentiates*; geocore's core automatically *verifies*. Every operator call runs its declared geometric invariants to machine precision — strict by default (`no_verify()` is the analogue of `torch.no_grad()`).

In [4]:
from geocore.invariants import no_verify, VerificationError

# every call above already self-checked silently.
# demonstrate the mechanism: register a deliberately wrong implementation
op_reg = get_op("pauli.commutes")

@op_reg.register(Pauli, Pauli)
def _wrong(a, b):
    return not a.commutes_with(b)   # sabotage

try:
    op_reg(Pauli("X"), Pauli("Z"))   # should raise: invariant violated
except VerificationError as e:
    print("caught by automatic verification:", e)
finally:
    del op_reg._implementations[(Pauli, Pauli)]

@op_reg.register(Pauli, Pauli)
def _correct(a, b):
    return a.commutes_with(b)

# no_verify() disables automatic verification (like torch.no_grad())
with no_verify():
    print("with no_verify(), the wrong impl passes silently:", op_reg(Pauli("X"), Pauli("Z")))

caught by automatic verification: pauli.commutes: invariant 'symplectic_form' failed: commutes(X,Z)=True but omega-form gives False
with no_verify(), the wrong impl passes silently: False


## L3 — reduce computation (≈ torch.compile)

The first shortcut is geometric in origin: `R_P(θ) = cos(θ/2) I − i sin(θ/2) P` because `P² = I` — the rotation orbit of the Pauli axis closes in two steps. This replaces the generic dense matrix-exponential path (O(8ⁿ)) with an O(2ⁿ) Pauli action. Every shortcut is **verified against the generic path to machine precision**, then **benchmarked**.

In [5]:
# verify: closed form == generic expm path, machine precision
r, s = Rotation("XYZ", 0.7), np.random.default_rng(0).standard_normal(8) + 1j*np.random.default_rng(1).standard_normal(8)
res, report = shortcuts.registry.apply("rotation.closed_form", r, s, verify=True)
print("machine-precision check vs generic path:", report.ok, f"(max error {report.max_error:.2e})")

machine-precision check vs generic path: True (max error 5.55e-17)


In [6]:
# measure the speedup (BenchmarkLog: time + FLOPs, no unmeasured claims)
rng = np.random.default_rng(0)
print(f"{'n':>3} {'time generic':>14} {'time shortcut':>14} {'speedup':>10} {'FLOPs speedup':>14}")
for n in [4, 6, 8, 10]:
    rot = Rotation("X"*n, 0.7)
    st = rng.standard_normal(2**n) + 1j*rng.standard_normal(2**n)
    log = shortcuts.registry.benchmark("rotation.closed_form", rot, st, n_trials=10,
                                       size_of=lambda a, b: len(a.axis))
    print(f"{n:>3} {log.time_generic:>14.2e} {log.time_shortcut:>14.2e} "
          f"{log.speedup_time:>10.1f}x {log.speedup_flops:>14.1e}x")

  n   time generic  time shortcut    speedup  FLOPs speedup
  4       3.21e-04       1.42e-05       22.6x        2.6e+02x
  6       1.26e-03       1.64e-05       77.0x        4.1e+03x


  8       2.58e-02       2.16e-05     1191.4x        6.6e+04x


 10       9.48e-01       2.97e-05    31927.1x        1.0e+06x


## Summary

The four layers mirror PyTorch's architecture, each with a geometric analogue:

| PyTorch | geocore |
|---|---|
| tensor engine | geometric objects (Pauli, Rotation) |
| aten/c10 dispatch | geometric operator dispatch |
| autograd | automatic verification (machine precision) |
| torch.compile | closed-form shortcuts with measured speedups |

Every number is measured, every shortcut is verified to machine precision before it is trusted. The theory is the engine, not the claim.